In [4]:
# =============================================================================
# Zelle 1: Alle notwendigen Imports
# =============================================================================

import os
import ast
import re
import numpy as np
import pandas as pd
from PIL import Image
from collections import defaultdict
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
import torchvision.transforms as transforms

from tqdm import tqdm
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"GPU Memory freigegeben.")
print(f"Belegt: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
print(f"Reserviert: {torch.cuda.memory_reserved() / 1024**2:.1f} MB")
print("PyTorch Version:", torch.__version__)
print("CUDA verfuegbar:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU Memory freigegeben.
Belegt: 0.0 MB
Reserviert: 0.0 MB
PyTorch Version: 1.12.1
CUDA verfuegbar: True
GPU: Tesla V100-SXM2-32GB


In [5]:
# =============================================================================
# Zelle 2: Pfade definieren, CSV laden, bbox parsen, Kamera/Zeitstempel extrahieren
# =============================================================================

BASE_DIR         = "/datasets/multi-view-pig-posture-recognition"
TRAIN_CSV_PATH   = os.path.join(BASE_DIR, "train2.csv")
TRAIN_IMAGES_DIR = os.path.join(BASE_DIR, "train2_images")
MODEL_SAVE_PATH  = os.path.join(BASE_DIR, "multiview_resnet50_best.pth")

# CSV laden -- ausschliesslich train2
df = pd.read_csv(TRAIN_CSV_PATH)
print(f"CSV geladen: {len(df)} Zeilen")
print(f"Spalten: {list(df.columns)}")

# --- bbox parsen ---
# bbox kann als String "[x, y, w, h]" vorliegen -> in Liste konvertieren
def parse_bbox(bbox_val):
    """Parst bbox aus String oder Liste in eine Liste [x, y, w, h]."""
    if isinstance(bbox_val, str):
        try:
            return ast.literal_eval(bbox_val)
        except (ValueError, SyntaxError):
            # Fallback: Zahlen per Regex extrahieren
            nums = re.findall(r"[\d.]+", bbox_val)
            return [float(n) for n in nums]
    elif isinstance(bbox_val, (list, tuple)):
        return list(bbox_val)
    else:
        return [0, 0, 1920, 1080]  # Fallback: ganzes Bild

df["bbox_parsed"] = df["bbox"].apply(parse_bbox)
df["bbox_x"] = df["bbox_parsed"].apply(lambda b: float(b[0]))
df["bbox_y"] = df["bbox_parsed"].apply(lambda b: float(b[1]))
df["bbox_w"] = df["bbox_parsed"].apply(lambda b: float(b[2]))
df["bbox_h"] = df["bbox_parsed"].apply(lambda b: float(b[3]))

# --- Kamera und Zeitstempel aus image_id extrahieren ---
# Format: pen1_orb_cam1_20250108_085204.jpg
# Strategie: Kamera = alles bis zum Datum, Zeitstempel = Datum+Uhrzeit-Teil

def extract_camera_and_timestamp(image_id):
    """
    Extrahiert Kamera-ID und Zeitstempel aus dem Dateinamen.
    Beispiel: 'pen1_orb_cam1_20250108_085204.jpg'
      -> camera: 'pen1_orb_cam1'
      -> timestamp: '20250108_085204'
    """
    name = os.path.splitext(image_id)[0]  # Endung entfernen
    # Suche nach dem Datums-Muster: 8 Ziffern gefolgt von _ und 6 Ziffern
    match = re.search(r"(\d{8}_\d{6})$", name)
    if match:
        timestamp = match.group(1)
        # Alles vor dem Zeitstempel ist die Kamera-ID (ohne trailing _)
        camera = name[:match.start()].rstrip("_")
        return camera, timestamp
    # Fallback: letzten zwei _-Segmente als Zeitstempel verwenden
    parts = name.rsplit("_", 2)
    if len(parts) >= 3:
        camera = parts[0]
        timestamp = parts[1] + "_" + parts[2]
        return camera, timestamp
    return name, "unknown"

df["camera"], df["timestamp"] = zip(*df["image_id"].apply(extract_camera_and_timestamp))

# --- Pen (Gehege) extrahieren fuer die Gruppierung ---
# Gleicher Zeitstempel + gleiches Gehege = ein Multi-View Sample
def extract_pen(image_id):
    """Extrahiert die Gehege-ID (z.B. 'pen1') aus dem Dateinamen."""
    match = re.match(r"(pen\d+)", image_id)
    return match.group(1) if match else "unknown_pen"

df["pen"] = df["image_id"].apply(extract_pen)

# Gruppierungsschluessel: Gehege + Zeitstempel + row-spezifische bbox
# WICHTIG: Ein Bild kann mehrere Schweine (bboxes) enthalten.
# Wir gruppieren nach pen + timestamp + class_id-Kombination, damit
# dasselbe Schwein aus verschiedenen Kameras zusammenkommt.
# Da wir die exakte Zuordnung nicht kennen, gruppieren wir pro
# pen + timestamp und behandeln jede bbox als eigenes Sample innerhalb
# der Gruppe. Die Multi-View-Aggregation erfolgt auf Bild-Ebene.

df["group_key"] = df["pen"] + "_" + df["timestamp"]

print(f"\nErkannte Kameras: {sorted(df['camera'].unique())}")
print(f"Erkannte Gehege: {sorted(df['pen'].unique())}")
print(f"Anzahl Klassen (class_id): {sorted(df['class_id'].unique())}")
print(f"Anzahl eindeutige Zeitstempel-Gruppen: {df['group_key'].nunique()}")
print(f"\nBeispiel:")
df.head(10)

CSV geladen: 23450 Zeilen
Spalten: ['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id']

Erkannte Kameras: ['pen1_orb_cam1', 'pen1_orb_cam2', 'pen1_tur_cam1', 'pen1_tur_cam2', 'pen2_orb_cam1', 'pen2_orb_cam2', 'pen2_tur_cam1', 'pen2_tur_cam2']
Erkannte Gehege: ['pen1', 'pen2']
Anzahl Klassen (class_id): [0, 1, 2, 3, 4]
Anzahl eindeutige Zeitstempel-Gruppen: 3084

Beispiel:


,row_id,image_id,width,height,bbox,class_id,bbox_parsed,bbox_x,bbox_y,bbox_w,bbox_h,camera,timestamp,pen,group_key
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0,"[1031.5, 368.0, 349.0, 435.0]",1031.5,368.0,349.0,435.0,pen1_orb_cam1,20250108_085204,pen1,pen1_20250108_085204
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4,"[1278.5, 428.0, 233.0, 438.0]",1278.5,428.0,233.0,438.0,pen1_orb_cam1,20250108_085204,pen1,pen1_20250108_085204
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1,"[732.0, 137.5, 342.0, 198.0]",732.0,137.5,342.0,198.0,pen1_orb_cam1,20250108_085204,pen1,pen1_20250108_085204
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0,"[830.0, 169.0, 370.0, 263.0]",830.0,169.0,370.0,263.0,pen1_orb_cam1,20250108_085204,pen1,pen1_20250108_085204
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3,"[611.5, 314.8, 381.5, 386.6]",611.5,314.8,381.5,386.6,pen1_orb_cam1,20250108_085204,pen1,pen1_20250108_085204
5,train_pen1_orb_cam1_20250108_085204_0005,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1023.4,261.6,340.6,192.4]",4,"[1023.4, 261.6, 340.6, 192.4]",1023.4,261.6,340.6,192.4,pen1_orb_cam1,20250108_085204,pen1,pen1_20250108_085204
6,train_pen1_orb_cam1_20250108_085204_0006,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[662.3,96.2,349.7,183.8]",1,"[662.3, 96.2, 349.7, 183.8]",662.3,96.2,349.7,183.8,pen1_orb_cam1,20250108_085204,pen1,pen1_20250108_085204
7,train_pen1_orb_cam1_20250108_085209_0000,pen1_orb_cam1_20250108_085209.jpg,1920,1080,"[1031.0,366.0,348.0,438.0]",0,"[1031.0, 366.0, 348.0, 438.0]",1031.0,366.0,348.0,438.0,pen1_orb_cam1,20250108_085209,pen1,pen1_20250108_085209
8,train_pen1_orb_cam1_20250108_085209_0001,pen1_orb_cam1_20250108_085209.jpg,1920,1080,"[800.5,169.0,397.0,267.0]",0,"[800.5, 169.0, 397.0, 267.0]",800.5,169.0,397.0,267.0,pen1_orb_cam1,20250108_085209,pen1,pen1_20250108_085209
9,train_pen1_orb_cam1_20250108_085209_0002,pen1_orb_cam1_20250108_085209.jpg,1920,1080,"[732.0,136.5,342.0,203.0]",1,"[732.0, 136.5, 342.0, 203.0]",732.0,136.5,342.0,203.0,pen1_orb_cam1,20250108_085209,pen1,pen1_20250108_085209


In [6]:
# =============================================================================
# Zelle 3: Daten inspizieren -- Gruppierung und Verteilung pruefen
# =============================================================================

# Wie viele Kameras pro Gruppe?
views_per_group = df.groupby("group_key")["camera"].nunique()
print("Kamera-Ansichten pro Gruppe (Verteilung):")
print(views_per_group.value_counts().sort_index())

# Klassen-Verteilung
print("\nKlassen-Verteilung (class_id):")
print(df["class_id"].value_counts().sort_index())

# Anzahl bboxes pro Bild
bboxes_per_image = df.groupby("image_id").size()
print(f"\nBboxes pro Bild -- Min: {bboxes_per_image.min()}, "
      f"Max: {bboxes_per_image.max()}, "
      f"Mittel: {bboxes_per_image.mean():.1f}")

# Kamera-Liste fuer das Dataset festlegen
CAMERA_LIST = sorted(df["camera"].unique())
NUM_VIEWS   = len(CAMERA_LIST)
NUM_CLASSES = df["class_id"].nunique()

# Label-Mapping (class_id -> fortlaufender Index)
unique_classes = sorted(df["class_id"].unique())
label_to_idx   = {cls: idx for idx, cls in enumerate(unique_classes)}
idx_to_label   = {idx: cls for cls, idx in label_to_idx.items()}

print(f"\nAnzahl Kamera-Ansichten (Views): {NUM_VIEWS}")
print(f"Kameras: {CAMERA_LIST}")
print(f"Anzahl Klassen: {NUM_CLASSES}")
print(f"Label-Mapping: {label_to_idx}")

Kamera-Ansichten pro Gruppe (Verteilung):
camera
1    3018
2      66
Name: count, dtype: int64

Klassen-Verteilung (class_id):
class_id
0    3083
1    3435
2     695
3    9928
4    6309
Name: count, dtype: int64

Bboxes pro Bild -- Min: 3, Max: 11, Mittel: 7.4

Anzahl Kamera-Ansichten (Views): 8
Kameras: ['pen1_orb_cam1', 'pen1_orb_cam2', 'pen1_tur_cam1', 'pen1_tur_cam2', 'pen2_orb_cam1', 'pen2_orb_cam2', 'pen2_tur_cam1', 'pen2_tur_cam2']
Anzahl Klassen: 5
Label-Mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}


In [7]:
# =============================================================================
# Zelle 4: Multi-View Dataset-Klasse mit Bounding-Box Crop
# =============================================================================

class MultiViewPigDataset(Dataset):
    """
    Dataset fuer Multi-View Schweine-Haltungserkennung.

    Jedes Sample besteht aus einem Schwein zu einem bestimmten Zeitpunkt.
    Das Dataset:
    1. Gruppiert Eintraege nach Gehege + Zeitstempel.
    2. Laedt fuer jede Kamera-Ansicht das zugehoerige Bild.
    3. Schneidet das Schwein anhand der Bounding Box zu (Crop).
    4. Skaliert den Crop auf 224x224 und normalisiert ihn.
    5. Gibt einen Tensor (NUM_VIEWS, C, H, W) plus Label zurueck.

    Falls ein Sample mehrere Bounding Boxes pro Bild hat (mehrere Schweine),
    wird jede Bbox als separates Sample pro Kameraansicht behandelt.
    Die Zuordnung erfolgt ueber den Index innerhalb der Gruppe.
    """

    def __init__(self, dataframe, images_dir, camera_list, label_map,
                 transform=None, crop_padding=20):
        super().__init__()
        self.images_dir  = images_dir
        self.camera_list = camera_list
        self.cam_to_idx  = {cam: i for i, cam in enumerate(camera_list)}
        self.transform   = transform
        self.label_map   = label_map
        self.crop_padding = crop_padding

        # Samples aufbauen:
        # Fuer jede Gruppe (pen+timestamp) sammeln wir die Bbox-Eintraege
        # pro Kamera. Da mehrere Schweine pro Bild existieren koennen,
        # erstellen wir pro Gruppe ein Sample pro Schwein-Index.
        self.samples = []

        grouped = dataframe.groupby("group_key")

        for group_key, group_df in grouped:
            # Innerhalb der Gruppe: pro Kamera die Bboxes sortiert sammeln
            cam_entries = defaultdict(list)
            for _, row in group_df.iterrows():
                cam = row["camera"]
                cam_entries[cam].append({
                    "image_id": row["image_id"],
                    "bbox_x": row["bbox_x"],
                    "bbox_y": row["bbox_y"],
                    "bbox_w": row["bbox_w"],
                    "bbox_h": row["bbox_h"],
                    "class_id": row["class_id"],
                })

            # Bestimme die maximale Anzahl Schweine (Bboxes) in einer Kamera
            max_bboxes = max(len(v) for v in cam_entries.values()) if cam_entries else 0

            for bbox_idx in range(max_bboxes):
                sample_views = {}
                sample_label = None

                for cam in camera_list:
                    entries = cam_entries.get(cam, [])
                    if bbox_idx < len(entries):
                        entry = entries[bbox_idx]
                        sample_views[cam] = entry
                        if sample_label is None:
                            sample_label = entry["class_id"]

                if sample_label is not None:
                    self.samples.append({
                        "views": sample_views,
                        "label": self.label_map[sample_label],
                    })

    def __len__(self):
        return len(self.samples)

    def _load_and_crop(self, entry):
        """
        Laedt ein Bild und schneidet es anhand der Bounding Box zu.
        Gibt das zugeschnittene PIL-Image zurueck.
        """
        img_path = os.path.join(self.images_dir, entry["image_id"])

        if os.path.isfile(img_path):
            img = Image.open(img_path).convert("RGB")
        else:
            # Fallback: schwarzes Bild
            return Image.new("RGB", (224, 224), (0, 0, 0))

        img_w, img_h = img.size
        pad = self.crop_padding

        # Bounding Box: [x, y, w, h] -> Crop-Koordinaten mit Padding
        x = int(entry["bbox_x"])
        y = int(entry["bbox_y"])
        w = int(entry["bbox_w"])
        h = int(entry["bbox_h"])

        left   = max(0, x - pad)
        top    = max(0, y - pad)
        right  = min(img_w, x + w + pad)
        bottom = min(img_h, y + h + pad)

        # Sicherheitscheck: Crop muss eine minimale Groesse haben
        if right <= left or bottom <= top:
            return Image.new("RGB", (224, 224), (0, 0, 0))

        cropped = img.crop((left, top, right, bottom))
        return cropped

    def __getitem__(self, idx):
        sample = self.samples[idx]
        views  = sample["views"]
        label  = sample["label"]

        images = []
        for cam in self.camera_list:
            if cam in views:
                img = self._load_and_crop(views[cam])
            else:
                # Fehlende Ansicht: schwarzes Bild als Platzhalter
                img = Image.new("RGB", (224, 224), (0, 0, 0))

            if self.transform:
                img = self.transform(img)
            images.append(img)

        # Stapeln zu (NUM_VIEWS, C, H, W)
        images_tensor = torch.stack(images, dim=0)
        return images_tensor, label


print(f"Dataset-Klasse definiert.")
print(f"  -- Laedt Bilder und schneidet Schweine per Bounding Box zu")
print(f"  -- Gruppiert nach Zeitstempel fuer Multi-View Aggregation")
print(f"  -- Fehlende Ansichten werden durch schwarze Platzhalter ersetzt")

Dataset-Klasse definiert.
  -- Laedt Bilder und schneidet Schweine per Bounding Box zu
  -- Gruppiert nach Zeitstempel fuer Multi-View Aggregation
  -- Fehlende Ansichten werden durch schwarze Platzhalter ersetzt


In [8]:
# =============================================================================
# Zelle 5: Transforms, Dataset auf 50% reduzieren und DataLoader
# =============================================================================

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

BATCH_SIZE  = 32
NUM_WORKERS = 10

# --- Nur 50% der Daten verwenden (stratifiziert nach class_id) ---
from sklearn.model_selection import train_test_split

df_half, _ = train_test_split(
    df,
    train_size=0.5,
    random_state=42,
    stratify=df["class_id"],
)
df_half = df_half.reset_index(drop=True)
print(f"Originale Daten: {len(df)} Zeilen")
print(f"Reduzierte Daten (50%): {len(df_half)} Zeilen")

train_dataset = MultiViewPigDataset(
    dataframe=df_half,
    images_dir=TRAIN_IMAGES_DIR,
    camera_list=CAMERA_LIST,
    label_map=label_to_idx,
    transform=train_transform,
    crop_padding=20,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

print(f"Dataset-Groesse: {len(train_dataset)} Samples")
print(f"Batches pro Epoche: {len(train_loader)}")
print(f"Geschaetzte Zeit pro Epoche: ~{len(train_loader) / 1.8:.0f} Sekunden")

Originale Daten: 23450 Zeilen
Reduzierte Daten (50%): 11725 Zeilen
Dataset-Groesse: 11519 Samples
Batches pro Epoche: 360
Geschaetzte Zeit pro Epoche: ~200 Sekunden


In [9]:
# =============================================================================
# Zelle 6: Multi-View ResNet-50 Architektur (Late Fusion mit Average-Pooling)
# =============================================================================

class MultiViewResNet50(nn.Module):
    """
    Multi-View Architektur auf Basis eines gemeinsamen ResNet-50 Backbones.

    Late-Fusion-Strategie:
    1. Eingabe: (B, V, C, H, W) mit V = Anzahl Kameraansichten.
    2. Jede Ansicht wird unabhaengig durch denselben ResNet-50 Feature-Extractor
       geleitet (alle Schichten ausser der originalen FC).
    3. Ergebnis: V Feature-Vektoren mit je 2048 Dimensionen pro Sample.
    4. Aggregation: Average-Pooling ueber alle V Ansichten -> ein 2048-d Vektor.
    5. Klassifikation: Der aggregierte Vektor geht durch den finalen
       Klassifikationskopf.

    Average-Pooling (statt Konkatenation) macht das Modell robust gegenueber
    fehlenden Ansichten und haelt die Parameteranzahl unabhaengig von V.
    """

    def __init__(self, num_views, num_classes, pretrained=True):
        super().__init__()
        self.num_views = num_views

        # Vortrainiertes ResNet-50 laden, originalen FC-Layer entfernen
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet50(weights=weights)
        self.feature_dim = backbone.fc.in_features  # 2048
        backbone.fc = nn.Identity()
        self.backbone = backbone

        # Klassifikationskopf auf dem aggregierten Feature-Vektor
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(self.feature_dim),
            nn.Dropout(0.4),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        """
        Args:
            x: Tensor der Form (B, V, C, H, W)
        Returns:
            logits: Tensor der Form (B, num_classes)
        """
        B, V, C, H, W = x.shape

        # Alle Ansichten auf einmal durch das Backbone schicken
        x = x.view(B * V, C, H, W)          # (B*V, C, H, W)
        features = self.backbone(x)          # (B*V, 2048)

        # Zurueck in View-Dimension aufteilen und Average-Pooling
        features = features.view(B, V, self.feature_dim)  # (B, V, 2048)
        features = features.mean(dim=1)                    # (B, 2048)

        logits = self.classifier(features)                 # (B, num_classes)
        return logits


print(f"Modell-Architektur: MultiViewResNet50 (Late Fusion, Average-Pooling)")
print(f"  Views: {NUM_VIEWS}")
print(f"  Klassen: {NUM_CLASSES}")
print(f"  Feature-Dim pro View: 2048")
print(f"  Aggregierte Feature-Dim: 2048 (Average-Pooling)")

Modell-Architektur: MultiViewResNet50 (Late Fusion, Average-Pooling)
  Views: 8
  Klassen: 5
  Feature-Dim pro View: 2048
  Aggregierte Feature-Dim: 2048 (Average-Pooling)


In [10]:
# =============================================================================
# Zelle 7: Modell instanziieren, Loss, Optimizer und Scheduler konfigurieren
# =============================================================================

DEVICE       = torch.device("cuda:0")
NUM_EPOCHS   = 10
LR           = 1e-4
WEIGHT_DECAY = 1e-4
TARGET_ACC   = 82.0  # Prozent

model = MultiViewResNet50(
    num_views=NUM_VIEWS,
    num_classes=NUM_CLASSES,
    pretrained=True,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Zusammenfassung
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Device:              {DEVICE}")
print(f"Epochen:             {NUM_EPOCHS}")
print(f"Learning Rate:       {LR}")
print(f"Weight Decay:        {WEIGHT_DECAY}")
print(f"Ziel-Accuracy:       {TARGET_ACC}%")
print(f"Batch Size:          {BATCH_SIZE}")
print(f"Parameter gesamt:    {total_params:,}")
print(f"Parameter trainbar:  {trainable_params:,}")

Device:              cuda:0
Epochen:             10
Learning Rate:       0.0001
Weight Decay:        0.0001
Ziel-Accuracy:       82.0%
Batch Size:          32
Parameter gesamt:    24,564,805
Parameter trainbar:  24,564,805


In [11]:
# =============================================================================
# Zelle 8: Trainingsschleife -- 20 Epochen, tqdm, Modellspeicherung bei 82%+
# =============================================================================

best_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss    = 0.0
    running_correct = 0
    running_total   = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoche {epoch:02d}/{NUM_EPOCHS}",
        leave=True,
        ncols=130,
    )

    for batch_idx, (images, labels) in enumerate(pbar):
        # images: (B, V, C, H, W), labels: (B,)
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        # Batch-Statistiken
        preds      = logits.argmax(dim=1)
        correct    = (preds == labels).sum().item()
        batch_size = labels.size(0)

        running_loss    += loss.item() * batch_size
        running_correct += correct
        running_total   += batch_size

        # tqdm dynamisch aktualisieren mit Live-Werten
        batch_acc = 100.0 * correct / batch_size
        avg_loss  = running_loss / running_total
        avg_acc   = 100.0 * running_correct / running_total
        pbar.set_postfix({
            "b_loss": f"{loss.item():.4f}",
            "b_acc":  f"{batch_acc:.1f}%",
            "avg_loss": f"{avg_loss:.4f}",
            "avg_acc":  f"{avg_acc:.1f}%",
        })

    # Epochen-Statistiken
    epoch_loss = running_loss / running_total
    epoch_acc  = 100.0 * running_correct / running_total

    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"  => Epoche {epoch:02d} abgeschlossen | "
          f"Loss: {epoch_loss:.4f} | "
          f"Accuracy: {epoch_acc:.2f}% | "
          f"LR: {current_lr:.6f}")



Epoche 01/10: 100%|█████████████████| 360/360 [04:25<00:00,  1.36it/s, b_loss=0.6601, b_acc=71.0%, avg_loss=0.9688, avg_acc=64.1%]


  => Epoche 01 abgeschlossen | Loss: 0.9688 | Accuracy: 64.12% | LR: 0.000098


Epoche 02/10: 100%|█████████████████| 360/360 [04:31<00:00,  1.32it/s, b_loss=0.4696, b_acc=83.9%, avg_loss=0.6595, avg_acc=75.7%]


  => Epoche 02 abgeschlossen | Loss: 0.6595 | Accuracy: 75.67% | LR: 0.000090


Epoche 03/10: 100%|█████████████████| 360/360 [04:28<00:00,  1.34it/s, b_loss=0.7192, b_acc=74.2%, avg_loss=0.5135, avg_acc=81.7%]


...

  => Epoche 08 abgeschlossen | Loss: 0.1752 | Accuracy: 94.00% | LR: 0.000010


Epoche 09/10: 100%|█████████████████| 360/360 [04:29<00:00,  1.33it/s, b_loss=0.1075, b_acc=96.8%, avg_loss=0.1439, avg_acc=95.4%]


  => Epoche 09 abgeschlossen | Loss: 0.1439 | Accuracy: 95.38% | LR: 0.000002


Epoche 10/10: 100%|█████████████████| 360/360 [04:32<00:00,  1.32it/s, b_loss=0.1117, b_acc=96.8%, avg_loss=0.1191, avg_acc=96.0%]

  => Epoche 10 abgeschlossen | Loss: 0.1191 | Accuracy: 96.04% | LR: 0.000000


In [12]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device="cuda:0")

# Speicherverbrauch prüfen
print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
print(f"Reserved:  {torch.cuda.memory_reserved(0) / 1024**2:.1f} MB")

Allocated: 522.8 MB
Reserved:  1748.0 MB
